First we try to divide the flight into 3 phases: Take off, Cruising, Landing

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, AutoDateLocator


# === Load all flights ===
with open("enriched_flight_data.json", "r") as f:
    flights_list = json.load(f)

def flatten_weather(entry):
    """Flatten nested 'weather' dictionary if it exists."""
    weather = entry.pop("weather", {})
    if weather:
        for k, v in weather.items():
            entry[k.replace(":", "_")] = v  # replace colons with underscores for JSON safety
    return entry

# === Segmentation function ===
def split_flight_by_altitude(df, tolerance_ratio=0.05):
    """Split flight into takeoff, cruise, landing based on altitude band."""
    df = df.sort_values('timestamp').reset_index(drop=True)
    alt = df['altitude_ft']
    alt_max = alt.max()
    lower = alt_max * (1 - tolerance_ratio)

    cruise_start_idx = alt[alt >= lower].index.min()
    cruise_end_idx = alt[alt >= lower].index.max()

    takeoff_df = df.loc[:cruise_start_idx] if cruise_start_idx > 0 else df.iloc[0:0]
    cruise_df = df.loc[cruise_start_idx:cruise_end_idx]
    landing_df = df.loc[cruise_end_idx:]
    return takeoff_df, cruise_df, landing_df

# === Apply segmentation ===

output_flights = []

total = len(flights_list)
print(f"Processing {total} flights...\n")

for i, flight_obj in enumerate(flights_list, start=1):
    metadata = flight_obj["flight_metadata"]
    data = flight_obj["flight_data"]

    if not data:
        print(f"Skipping flight {i}: no data.")
        continue

    # Flatten weather data
    flattened_data = [flatten_weather(d.copy()) for d in data]

    df = pd.DataFrame(flattened_data)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    df.rename(columns={"alt": "altitude_ft"}, inplace=True)

    takeoff_df, cruise_df, landing_df = split_flight_by_altitude(df)

    # --- Convert timestamps to string for JSON ---
    for segment in [takeoff_df, cruise_df, landing_df]:
        segment["timestamp"] = segment["timestamp"].astype(str)

    output_flights.append({
        "flight_metadata": metadata,
        "take_off": takeoff_df.to_dict(orient="records"),
        "cruising": cruise_df.to_dict(orient="records"),
        "landing": landing_df.to_dict(orient="records")
    })

    print(f"Processed flight {i}/{total}: {metadata.get('flight', 'Unknown')}")

# === Save new file ===
output_path = "flight_data_separated_into_phases.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_flights, f, indent=4)

print(f"\n✅ All done! Saved to {output_path}")


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, AutoDateLocator

# === Load the phase-separated flight data ===
with open("flight_data_separated_into_phases.json", "r", encoding="utf-8") as f:
    flights = json.load(f)

# === Take the first flight only ===
flight = flights[0]
meta = flight["flight_metadata"]
print(f"Plotting flight: {meta.get('flight')} ({meta.get('fr24_id')})")

# === Convert each phase into a DataFrame ===
def df_from_phase(phase_data):
    if not phase_data:
        return pd.DataFrame(columns=["timestamp", "altitude_ft"])
    df = pd.DataFrame(phase_data)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    if "altitude_ft" not in df.columns and "alt" in df.columns:
        df.rename(columns={"alt": "altitude_ft"}, inplace=True)
    return df.sort_values("timestamp")

takeoff_df = df_from_phase(flight.get("take_off", []))
cruise_df  = df_from_phase(flight.get("cruising", []))
landing_df = df_from_phase(flight.get("landing", []))

# === Plot altitude vs time with different colors ===
plt.figure(figsize=(12, 6))

if not takeoff_df.empty:
    plt.plot(takeoff_df["timestamp"], takeoff_df["altitude_ft"],
             color="gold", label="Take-off Phase", linewidth=2)
if not cruise_df.empty:
    plt.plot(cruise_df["timestamp"], cruise_df["altitude_ft"],
             color="skyblue", label="Cruising Phase", linewidth=2)
if not landing_df.empty:
    plt.plot(landing_df["timestamp"], landing_df["altitude_ft"],
             color="tomato", label="Landing Phase", linewidth=2)

# === Formatting ===
plt.title(f"Flight {meta.get('flight')} — Altitude Segmentation by Phase")
plt.xlabel("Time (UTC)")
plt.ylabel("Altitude (ft)")
plt.legend()
plt.grid(True)

locator = AutoDateLocator()
formatter = DateFormatter("%H:%M")
plt.gca().xaxis.set_major_locator(locator)
plt.gca().xaxis.set_major_formatter(formatter)

plt.tight_layout()
plt.show()


LETS COMPUTE THE TEXTUAL REPRESENTATION OF EACH PHASES

In [ ]:
!pip install openai==0.28

In [ ]:
import re

# === JSON cleanup ===
def clean_answer(raw_response: str):
    """
    Cleans and safely parses LLM JSON output (even if wrapped in markdown fences).
    """
    # Remove Markdown code fences (```json ... ```)
    cleaned = re.sub(r"^```(?:json|python)?", "", raw_response.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"```$", "", cleaned.strip())

    # Remove common markdown artifacts
    cleaned = cleaned.replace("\n", "\\n").replace("\r", "\\r")

    # Some models may prepend language tag or commentary
    cleaned = re.sub(r"^json", "", cleaned.strip(), flags=re.IGNORECASE)

    # Extra cleanup for accidental trailing commas
    cleaned = re.sub(r",\s*([\]\}])", r"\1", cleaned)

    # Try JSON parsing
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        # print(f"⚠️ JSON parsing failed, attempting manual recovery: {e}")
        # # Attempt a secondary cleaning
        cleaned2 = cleaned.replace('\\n', ' ').replace('\\"', '"')
        return json.loads(cleaned2)

In [ ]:
import os
import json
from datetime import datetime
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

# === Model definition ===
# chat_model = ChatOpenAI(model="gpt-4o-2024-11-20", openai_api_key=os.getenv("OPENAI_API_KEY_2"), temperature=0.3)

chat_model = ChatOpenAI(model="gpt-4.1", openai_api_key=os.getenv("OPENAI_API_KEY_2"), temperature=0.3)

# === Prompt (phase-aware, time-referenced, smooth transitions) ===
prompt_phase = """
You are given a segment of flight data corresponding to one phase of a flight 
(either takeoff, cruise, or landing). The data is structured as a list of datapoints, 
each containing the aircraft’s state and environmental conditions at a given time.

Each datapoint is a dictionary with keys such as:
- lat, lon, track, altitude_ft, gspeed (knots), vspeed (ft/min), minutes_since_start (float)
- edr_300hPa, ellrod_300hPa, richardson_number_300hPa, cape,
- wind_speed_10000m, wind_dir_10000m, wind_speed_2m, wind_dir_2m.

Your task is to write a **continuous, expert-level narrative** describing this phase of flight, 
progressing through the datapoints chronologically.

For each datapoint:
1. Use the field `minutes_since_start` to reference time relative to takeoff.
   Example: “5 minutes after departure” (for takeoff) or “5 minutes into the flight” (for cruise/landing).
2. Describe the aircraft's position (lat/lon) and motion (heading, altitude, ground speed, vertical speed).
3. Include meteorological context: turbulence (EDR, Ellrod), stability (Richardson number), CAPE, and wind information.
4. Mention relevant geographic regions based on coordinates (e.g., cities, monuments, regions, landscapes, montains, seas, rivers etc...).
5. Use **smooth transitions** between datapoints:
   - “Then the aircraft turns east by 20° and climbs another 300 ft...”
   - “A few minutes later, it begins a gentle descent...”
   - “As it approaches the Tyrrhenian coast...”
6. Do not forget to include the plane's type at the begining of your description: {type}

Write the result as a natural, continuous, and professional description, not as bullet points. I want to description to be very detailed.

Here is the flight phase's record:
{phase_data}

Return your answer as a JSON list containing the following 3 elements:
["{flight_id}", "{phase_name}", "The flight phase's narrative"]
"""

phase_prompt = ChatPromptTemplate.from_template(prompt_phase)

# === LLM chain definition ===
phase_summary_chain = (
    {
        "type": lambda x: x["type"],
        "phase_data": lambda x: x["data"],
        "flight_id": lambda x: x["fr24_id"],
        "phase_name": lambda x: x["phase"]
    }
    | phase_prompt
    | chat_model
    | StrOutputParser()
)

# === Helper to compute minutes since departure ===
def add_minutes_since_start(phase_data):
    if not phase_data:
        return []
    try:
        timestamps = [datetime.fromisoformat(p["timestamp"].replace("Z", "+00:00")) for p in phase_data]
        t0 = timestamps[0]
        for p, t in zip(phase_data, timestamps):
            delta_min = (t - t0).total_seconds() / 60.0
            p["minutes_since_start"] = round(delta_min, 1)
    except Exception:
        # In case of missing or bad timestamps
        for p in phase_data:
            p["minutes_since_start"] = 0.0
    return phase_data


# === Load dataset ===
with open("data/CDG-FCO/flight_data_separated_into_phases.json", "r") as f:
    flights_data = json.load(f)

# === Parameters ===
BATCH_SIZE = 15
total = len(flights_data)
print(f"Processing {total} flights...\n")

all_results = []
failed = []

# === Process flights in batches ===
for batch_start in range(0, total, BATCH_SIZE):
    batch = flights_data[batch_start:batch_start + BATCH_SIZE]
    print(f"🧩 Batch {(batch_start // BATCH_SIZE) + 1}: flights {batch_start + 1}–{batch_start + len(batch)}")

    # Step 1. Build the list of phase requests
    phase_requests = []
    flight_index = {}  # To map (flight_id, phase) back to all_results

    for flight in batch:
        fr24_id = flight["flight_metadata"]["fr24_id"]
        type = flight["flight_metadata"]["type"]
        for phase_name in ["take_off", "cruising", "landing"]:
            phase_data = add_minutes_since_start(flight.get(phase_name, []))
            if not phase_data:
                continue
            req = {"type": type, "phase": phase_name, "data": phase_data, "fr24_id": fr24_id}
            phase_requests.append(req)
            flight_index[(fr24_id, phase_name)] = None  # placeholder

    print(f"  🚀 Sending {len(phase_requests)} phase requests to GPT...")

    # Step 2. Send all at once
    try:
        responses = phase_summary_chain.batch(phase_requests, {"max_concurrency": BATCH_SIZE*3})
    except Exception as e:
        print(f"  ❌ Batch request failed: {e}")
        failed.extend([(r['fr24_id'], r['phase']) for r in phase_requests])
        continue

    # Step 3. Parse each response and assign to the correct (flight, phase)
    for raw_response, req in zip(responses, phase_requests):
        fr24_id, phase_name = req["fr24_id"], req["phase"]
        try:
            parsed = clean_answer(raw_response)
            # expected [flight_id, phase_name, narrative]
            narrative = parsed[2]
            flight_index[(fr24_id, phase_name)] = narrative
            print(f"  ✅ Parsed {fr24_id} ({phase_name})")
        except Exception as e:
            print(f"  ⚠️ Parsing failed for {fr24_id} ({phase_name}): {e}")
            failed.append((fr24_id, phase_name))

    # Step 4. Assemble results for flights in this batch
    for flight in batch:
        fr24_id = flight["flight_metadata"]["fr24_id"]
        result_entry = {
            "fr24_id": fr24_id,
            "take_off": flight_index.get((fr24_id, "take_off"), "No data or failed."),
            "cruising": flight_index.get((fr24_id, "cruising"), "No data or failed."),
            "landing": flight_index.get((fr24_id, "landing"), "No data or failed.")
        }
        all_results.append(result_entry)

    print(f"✅ Completed batch {(batch_start // BATCH_SIZE) + 1}\n")

# === Save output ===
output_path = "data/CDG-FCO/flight_summary_separated_into_phases.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\n✅ All flights processed. Results saved to {output_path}")
if failed:
    print(f"⚠️ {len(failed)} failed phases: {failed}")


In [ ]:
!pip install --upgrade openai

In [ ]:
# !pip install openai faiss-cpu matplotlib

import os
import json
import faiss
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from openai import OpenAI

# --------------------------
# Config
# --------------------------
MODEL = "text-embedding-3-large"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))
K = 5  # number of most similar/dissimilar flights
PHASES = ["take_off", "cruising", "landing"]

# --------------------------
# Load summaries and data
# --------------------------
with open("../data/CDG-FCO/flight_summary_separated_into_phases.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

with open("../data/CDG-FCO/flight_data_separated_into_phases.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

# Map fr24_id -> raw data per phase
data_by_id = {f["flight_metadata"]["fr24_id"]: f for f in flights_data}

# Map fr24_id -> summaries per phase, safely handling nulls
summary_by_phase = {}
for s in summaries:
    fid = s["fr24_id"]
    summary_by_phase[fid] = {}
    for phase in PHASES:
        text = s.get(phase)
        summary_by_phase[fid][phase] = text if isinstance(text, str) and text.strip() else None

common_ids = list(set(summary_by_phase.keys()) & set(data_by_id.keys()))
print(f"✅ Common flights: {len(common_ids)}")

# --------------------------
# Embeddings helper
# --------------------------
def get_embeddings(texts, model=MODEL, batch_size=64):
    vectors = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        batch_vecs = [np.array(e.embedding, dtype="float32") for e in resp.data]
        vectors.extend(batch_vecs)
    X = np.vstack(vectors)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)  # L2 normalization
    return X

# --------------------------
# Build FAISS index per phase (robust to missing summaries)
# --------------------------
phase_indices = {}
phase_metas = {}

for phase in PHASES:
    print(f"\n🧠 Building FAISS index for phase: {phase}")
    valid_flights = [fid for fid in common_ids if summary_by_phase[fid].get(phase)]
    texts = [summary_by_phase[fid][phase] for fid in valid_flights]
    if not texts:
        print(f"⚠️ No valid flights for phase {phase}, skipping index.")
        continue

    E = get_embeddings(texts)
    d = E.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(E)
    phase_indices[phase] = index
    phase_metas[phase] = valid_flights
    print(f"✅ Phase '{phase}': added {len(valid_flights)} valid flights.")

print("\n✅ All FAISS indices built successfully.\n")

# --------------------------
# Compute similarities per phase
# --------------------------
def find_similar_and_dissimilar(target_id, phase, k=K):
    """Find top-k and bottom-k flights for a given phase."""
    if phase not in phase_indices:
        raise ValueError(f"No index built for phase '{phase}'.")
    if not summary_by_phase[target_id].get(phase):
        raise ValueError(f"Target flight {target_id} has no data for phase '{phase}'.")

    q_vec = get_embeddings([summary_by_phase[target_id][phase]])
    index = phase_indices[phase]
    sims, idxs = index.search(q_vec, len(phase_metas[phase]))
    sims, idxs = sims[0], idxs[0]

    pairs = [
        (phase_metas[phase][i], float(sims[i]))
        for i in range(len(sims))
        if phase_metas[phase][i] != target_id
    ]
    pairs.sort(key=lambda x: x[1], reverse=True)
    return pairs[:k], pairs[-k:], pairs

# --------------------------
# Helpers for plotting
# --------------------------
def parse_time(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))

def extract_phase_data(fid, phase):
    phase_data = data_by_id[fid].get(phase, [])
    if not phase_data:
        return [], [], [], []
    times = [parse_time(d["timestamp"]) for d in phase_data]
    alts = [d.get("altitude_ft", 0) for d in phase_data]
    lats = [d.get("lat", 0) for d in phase_data]
    lons = [d.get("lon", 0) for d in phase_data]
    return times, alts, lats, lons

def plot_altitude_phase(target_id, compare_ids, compare_sims, phase):
    plt.figure(figsize=(10,6))
    t, v, *_ = extract_phase_data(target_id, phase)
    if not t:
        print(f"⚠️ No altitude data for target {target_id} phase {phase}")
        return
    plt.plot(t, v, label=f"Target {target_id}", linewidth=2, color="black")
    for cid, sim in zip(compare_ids, compare_sims):
        t, v, *_ = extract_phase_data(cid, phase)
        if not t: 
            continue
        plt.plot(t, v, linestyle="--", label=f"{cid} (sim={sim:.2f})")
    plt.title(f"{phase.capitalize()} Phase: Altitude Comparison")
    plt.ylabel("Altitude (ft)")
    plt.xlabel("Time (UTC)")
    plt.legend()
    plt.show()

def plot_trajectory_phase(target_id, compare_ids, compare_sims, phase):
    plt.figure(figsize=(8,8))
    *_, lat, lon = extract_phase_data(target_id, phase)
    if not lat:
        print(f"⚠️ No trajectory data for target {target_id} phase {phase}")
        return
    plt.plot(lon, lat, label=f"Target {target_id}", linewidth=2, color="black")
    for cid, sim in zip(compare_ids, compare_sims):
        *_, lat, lon = extract_phase_data(cid, phase)
        if not lat:
            continue
        plt.plot(lon, lat, linestyle="--", label=f"{cid} (sim={sim:.2f})")
    plt.title(f"{phase.capitalize()} Phase: Trajectories")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.legend()
    plt.show()

# --------------------------
# Identify valid target flights
# --------------------------
valid_targets = [
    fid for fid, phases in summary_by_phase.items()
    if all(phases.get(p) for p in PHASES)
]
print(f"\n✅ Valid target flights (with all 3 phases): {len(valid_targets)}")

if not valid_targets:
    raise ValueError("No complete flights found with all phases!")


In [ ]:
target_id = valid_targets[3]
print(f"🎯 Target flight: {target_id}\n")

# --------------------------
# Similarity + Visualization per phase
# --------------------------

for phase in [PHASES[2]]:
    if phase not in phase_indices:
        print(f"⚠️ Skipping {phase}: no index built.")
        continue
    print(f"🔹 Analyzing phase: {phase}")
    try:
        top, bottom, ranking = find_similar_and_dissimilar(target_id, phase, k=K)
    except ValueError as e:
        print(f"⚠️ {e}")
        continue

    if not top or not bottom:
        print(f"⚠️ Not enough data for phase {phase}.")
        continue

    top_ids, top_sims = zip(*top)
    bottom_ids, bottom_sims = zip(*bottom)

    print(f"Top-{K} similar flights ({phase}): {top_ids}")
    print(f"Bottom-{K} dissimilar flights ({phase}): {bottom_ids}")

    plot_trajectory_phase(target_id, top_ids, top_sims, phase)
    plot_trajectory_phase(target_id, bottom_ids, bottom_sims, phase)

WHAT WE WANT TO DO NOW IS THE MITIGATION PART.

In [ ]:
import json
from datetime import datetime
import copy

def enrich_with_minutes_since_start(phase_data):
    """Ajoute le champ minutes_since_start basé sur le premier timestamp de la phase."""
    if not phase_data or len(phase_data) < 1:
        return []

    enriched = copy.deepcopy(phase_data)
    t0 = datetime.fromisoformat(enriched[0]["timestamp"].replace("Z", "+00:00"))

    for dp in enriched:
        t = datetime.fromisoformat(dp["timestamp"].replace("Z", "+00:00"))
        dp["minutes_since_start"] = round((t - t0).total_seconds() / 60, 2)

    return enriched

with open("flight_data_separated_into_phases.json", "r", encoding="utf-8") as f:
    flights = json.load(f)


for flight in flights:
    for phase in ["take_off", "cruising", "landing"]:
        if phase in flight and isinstance(flight[phase], list):
            flight[phase] = enrich_with_minutes_since_start(flight[phase])


In [ ]:
output_path = "flight_data_with_minutes_since_start.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(flights, f, indent=2, ensure_ascii=False)

print(f"✅ Enriched data saved in {output_path}")

=========================================================================================
RUN THE CODE FROM HERE
=========================================================================================

In [ ]:
import json
from openai import OpenAI
import os
import faiss
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

# === Configuration ===
MODEL = "gpt-4.1"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))
PHASES = ["take_off", "cruising", "landing"]

# === Prompt ===
PROMPT_PHASE_TEMPLATE = """
You are given a segment of flight data corresponding to the {phase_name} phase of a flight. The data is structured as a list of datapoints, 
each containing the aircraft’s state and environmental conditions at a given time.

Each datapoint is a dictionary with keys such as:
- lat, lon, track, altitude_ft, gspeed (knots), vspeed (ft/min), minutes_since_start (float)
- edr_300hPa, ellrod_300hPa, richardson_number_300hPa, cape,
- wind_speed_10000m, wind_dir_10000m, wind_speed_2m, wind_dir_2m.

Your task is to write a **continuous, expert-level narrative** describing this phase of flight, 
progressing through the datapoints chronologically.

For each datapoint:
1. Use the field `minutes_since_start` to reference time relative to takeoff.
   Example: “5 minutes after departure” (for takeoff) or “5 minutes into the flight” (for cruise/landing).
2. Describe the aircraft's position (lat/lon) and motion (heading, altitude, ground speed, vertical speed).
3. Include meteorological context: turbulence (EDR, Ellrod), stability (Richardson number), CAPE, and wind information.
4. Mention relevant geographic regions based on coordinates (e.g., cities, monuments, regions, landscapes, montains, seas, rivers etc...).
5. Use **smooth transitions** between datapoints:
   - “Then the aircraft turns east by 20° and climbs another 300 ft...”
   - “A few minutes later, it begins a gentle descent...”
   - “As it approaches the Tyrrhenian coast...”
6. Do not forget to include the plane's type at the begining of your description: {type}

Write the result as a natural, continuous, and professional description, not as bullet points. I want to description to be very detailed.

Here is the flight phase's record:
{phase_data}

Return the flights summary without any metatext"""

# === Generate flight summary for N datapoints ===
def generate_summary(phase_name, phase_data, aircraft_type, target_flight_id):

    prompt = PROMPT_PHASE_TEMPLATE.format(
        phase_name=phase_name,
        phase_data=json.dumps(phase_data),
        type=aircraft_type
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    parsed = response.choices[0].message.content.strip()
    return parsed # narrative only



# === build final prediction prompt ===
def build_prediction_prompt(target_summary, similar_summaries, N):
    prompt = f"""The target flight entered a GPS spoofed zone. Here is its trajectory summary before spoofing:
{target_summary}

Here are the summaries of the 5 most similar flights:"""
    for i, summ in enumerate(similar_summaries):
        prompt += f"\n\nSimilar flight {i+1}:\n{summ}"

    prompt += f"""

Based on these flights, where will the target flight be {N} minutes after the start of the phase?

⚠️ Return only the predicted position in this exact format:
LATITUDE, LONGITUDE
"""
    return prompt

# === Calls model for prediction ===
def predict_position(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [ ]:
import os
import json
import faiss
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))

# --------------------------
# CONFIGURATION
# --------------------------
EMBED_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4.1"
K = 6  # number of similar flights retrieved
PHASE = "take_off"  # phase we test
EMBEDDINGS_PATH = "../data/MULTI_ROUTE/EMBEDDINGS"

# --------------------------
# LOAD DATA
# --------------------------
with open("../data/CDG-FCO/flight_data_with_minutes_since_start.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

with open("../data/CDG-FCO/flight_summary_separated_into_phases.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

data_by_id = {f["flight_metadata"]["fr24_id"]: f for f in flights_data}
summary_by_phase = {
    s["fr24_id"]: {p: s.get(p, None) for p in ["take_off", "cruising", "landing"]}
    for s in summaries
}

print(f"✅ Loaded {len(flights_data)} flights with data and summaries.\n")


def save_phase_index(phase, index, ids, embeddings):
    faiss.write_index(index, f"{EMBEDDINGS_PATH}/faiss_index_{phase}.index")
    np.save(f"{EMBEDDINGS_PATH}/embeddings_{phase}.npy", embeddings)
    with open(f"{EMBEDDINGS_PATH}/ids_{phase}.json", "w") as f:
        json.dump(ids, f)
    print(f"💾 Saved FAISS index, embeddings and IDs for phase '{phase}'")


def load_phase_index(phase):
    index = faiss.read_index(f"{EMBEDDINGS_PATH}/faiss_index_{phase}.index")
    embeddings = np.load(f"{EMBEDDINGS_PATH}/embeddings_{phase}.npy")
    with open(f"{EMBEDDINGS_PATH}/ids_{phase}.json", "r") as f:
        ids = json.load(f)
    print(f"📂 Loaded FAISS index, embeddings and IDs for phase '{phase}'")
    return index, ids, embeddings

# --------------------------
# EMBEDDING HELPER
# --------------------------
def get_embeddings(texts, model=EMBED_MODEL, batch_size=32):
    """Embed multiple texts with normalization."""
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        embs = [np.array(e.embedding, dtype="float32") for e in resp.data]
        vecs.extend(embs)
    X = np.vstack(vecs)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    return X

# --------------------------
# BUILD RAG INDEX FOR EACH PHASE
# --------------------------

def build_phase_index(phase, force_rebuild=False):
    # 1) If files exist → load them
    if (not force_rebuild and 
        os.path.exists(f"{EMBEDDINGS_PATH}/faiss_index_{phase}.index") and
        os.path.exists(f"{EMBEDDINGS_PATH}/embeddings_{phase}.npy") and
        os.path.exists(f"{EMBEDDINGS_PATH}/ids_{phase}.json")):

        return load_phase_index(phase)

    # 2) Else → rebuild the index
    valid = [fid for fid, s in summary_by_phase.items() if s.get(phase)]
    texts = [summary_by_phase[fid][phase] for fid in valid]

    if not texts:
        raise ValueError(f"No valid summaries for phase '{phase}'")

    E = get_embeddings(texts)
    d = E.shape[1]

    index = faiss.IndexFlatIP(d)
    index.add(E)

    print(f"🔧 Index built for phase '{phase}' with {len(valid)} flights")

    # 3) Save everything for next time
    save_phase_index(phase, index, valid, E)

    return index, valid, E

In [ ]:
phase_index_take_off, phase_ids_take_off, _ = build_phase_index("take_off")

In [ ]:
phase_index_cruising, phase_ids_cruising, _ = build_phase_index("cruising")

In [ ]:
phase_index_landing, phase_ids_landing, _ = build_phase_index("landing")

In [ ]:
index_dictionary = {"take_off": phase_index_take_off, "cruising": phase_index_cruising, "landing": phase_index_landing}
ids_dictionary = {"take_off": phase_ids_take_off, "cruising": phase_ids_cruising, "landing": phase_ids_landing}

In [ ]:
import re

# === JSON cleanup ===
def clean_answer(raw_response: str):
    """
    Cleans and safely parses LLM JSON output (even if wrapped in markdown fences).
    """
    # Remove Markdown code fences (```json ... ```)
    cleaned = re.sub(r"^```(?:json|python)?", "", raw_response.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"```$", "", cleaned.strip())

    # Remove common markdown artifacts
    cleaned = cleaned.replace("\n", "\\n").replace("\r", "\\r")

    # Some models may prepend language tag or commentary
    cleaned = re.sub(r"^json", "", cleaned.strip(), flags=re.IGNORECASE)

    # Extra cleanup for accidental trailing commas
    cleaned = re.sub(r",\s*([\]\}])", r"\1", cleaned)

    # Try JSON parsing
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        # print(f"⚠️ JSON parsing failed, attempting manual recovery: {e}")
        # # Attempt a secondary cleaning
        cleaned2 = cleaned.replace('\\n', ' ').replace('\\"', '"')
        return json.loads(cleaned2)

In [ ]:
PREDICT_POSITION_PROMPT = f"""
A flight entered a GPS spoofing zone. Therefore his position is now unknown and your task is to help the pilot 
by predicting the flight's latitude and longitude.
Here is its known trajectory (before spoofing):

{{partial_summary}}

The following are summaries of {{number_similar_flight}} similar flights for the same phase:

{{sim_text}}

{{predictions_history}}

Based on these, predict the geographic position of the target flight {{predict_minutes}} minutes after the beginning of the phase.
Return your answer as JSON: {{{{"lat": "the predicted latitude", "lon": "the predicted longitude"}}}} only.
"""

# --------------------------
# PREDICT POSITION VIA LLM
# --------------------------
def predict_position_from_prompt(partial_summary, similar_summaries, predict_minutes, prediction_history: list=[]):
    """Ask GPT to predict target position after N minutes using similar flights."""
    sim_text = "\n\n".join([f"Similar flight {i+1}:\n{txt}" for i, txt in enumerate(similar_summaries)])
    if not prediction_history:
        predictions_history = ""
    else:
        predictions_history = f"""Here are the predictions you already made. They are given as a list of triplet [latitude, longitude, number of minutes into the phase]. Use these past predictions to ensure that your next prediction is consistent.
Make sure the plane continues moving forward along its trajectory, does not remain at the same coordinates repeatedly, and does not move backward in a way that violates realistic flight dynamics.

PREDICTION HISTORY: {{prediction_history}}"""
        predictions_history = predictions_history.format(prediction_history=prediction_history)

    prompt = PREDICT_POSITION_PROMPT.format(
        partial_summary=partial_summary, 
        number_similar_flight=len(similar_summaries),
        sim_text=sim_text,
        predict_minutes=predict_minutes,
        predictions_history=predictions_history,
    )

    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
        
    return clean_answer(response.choices[0].message.content), response.usage.total_tokens


In [ ]:
# --------------------------
# FIND SIMILAR FLIGHTS
# --------------------------
def find_similar_flights(partial_summary, target_id, phase, k=K, batch_size=None):
    """Embed the target partial summary and find K most similar full summaries."""
    q_vec = get_embeddings([partial_summary])
    # if batch_size:
    #     index = faiss.read_index(f"data/CDG-FCO/sub_indexes/faiss_index_{phase}_{batch_size}.index")
    #     sims, idxs = index.search(q_vec, k)
    #     ids_dictionary = {
    #         "take_off": [fid for fid, s in summary_by_phase.items() if s.get("take_off")], 
    #         "cruising": [fid for fid, s in summary_by_phase.items() if s.get("cruising")], 
    #         "landing": [fid for fid, s in summary_by_phase.items() if s.get("landing")]
    #     }
    # else:
    #     sims, idxs = index_dictionary[phase].search(q_vec, k)
    sims, idxs = index_dictionary[phase].search(q_vec, k)
    ids_for_phase = ids_dictionary[phase]
    sims, idxs = sims[0], idxs[0]
    neighbors = [(ids_for_phase[i], float(sims[i])) for i in range(len(idxs)) if ids_for_phase[i] != target_id]
    return neighbors


NOW WE WILL RUN THE COMPLETE EXPERIMENT 

In [ ]:
import folium
print(folium.__version__)

In [ ]:
import os
import json

def save_RAG_results(json_path, phase_name, observed, ground_truth, predicted, mean_error, mean_similar_flight):
    """
    Stores RAG predictions for a given phase inside a JSON file.

    json_path   : path to JSON file
    phase_name  : "takeoff", "cruising", "landing"
    observed    : numpy array of observed points
    ground_truth: numpy array of GT points
    predicted   : numpy array of predicted points
    mean_error  : float
    """

    # Load existing JSON file if present
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            results = json.load(f)
    else:
        results = {}

    # Insert or update section for the phase
    parsed_observed_data = [[point["lat"], point["lon"], point["minutes_since_start"]] for point in observed]
    results[phase_name] = {
        "observed": parsed_observed_data,
        "ground_truth": ground_truth,
        "predicted": predicted,
        "mean_similar_flight": mean_similar_flight,
        "mean_haversine_error": float(mean_error)
    }

    # Save back to disk
    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)

    print(f"📁 Saved results for phase '{phase_name}' into {json_path}")


In [ ]:
import math

EARTH_RADIUS = 6371000  # meters
EPSILON = 0.3

def latlon_to_local_xy(lat, lon, ref_lat, ref_lon):
    """
    Convert lat/lon to local Cartesian coordinates (meters)
    centered at ref_lat/ref_lon.
    """
    lat_rad = math.radians(lat)
    lon_rad = math.radians(lon)
    ref_lat_rad = math.radians(ref_lat)
    ref_lon_rad = math.radians(ref_lon)

    x = EARTH_RADIUS * (lon_rad - ref_lon_rad) * math.cos(ref_lat_rad)
    y = EARTH_RADIUS * (lat_rad - ref_lat_rad)

    return x, y


def local_xy_to_latlon(x, y, ref_lat, ref_lon):
    """
    Convert local Cartesian coordinates (meters)
    back to lat/lon.
    """
    ref_lat_rad = math.radians(ref_lat)
    ref_lon_rad = math.radians(ref_lon)

    lat_rad = y / EARTH_RADIUS + ref_lat_rad
    lon_rad = x / (EARTH_RADIUS * math.cos(ref_lat_rad)) + ref_lon_rad

    return math.degrees(lat_rad), math.degrees(lon_rad)


def enforce_ring_constraint_cartesian(prev_lat, prev_lon,
                                      pred_lat, pred_lon,
                                      expected_distance_m,
                                      epsilon=0.3):

    # Convert predicted point into local coordinates
    x, y = latlon_to_local_xy(pred_lat, pred_lon, prev_lat, prev_lon)

    dist = math.sqrt(x**2 + y**2)

    d_min = (1 - epsilon) * expected_distance_m
    d_max = (1 + epsilon) * expected_distance_m

    # Already inside ring
    if d_min <= dist <= d_max:
        return pred_lat, pred_lon, False

    if dist == 0:
        return pred_lat, pred_lon, False

    # Choose correct boundary
    if dist < d_min:
        target_dist = d_min
    else:
        target_dist = d_max

    ratio = target_dist / dist

    x_corrected = x * ratio
    y_corrected = y * ratio

    corrected_lat, corrected_lon = local_xy_to_latlon(
        x_corrected, y_corrected, prev_lat, prev_lon
    )

    return corrected_lat, corrected_lon, True


def get_mean_speed_similar_flights(similar_ids, data_by_id, phase, t_min):
    """
    Returns mean horizontal speed (m/s)
    from similar flights at time t_min.
    """
    speeds = []

    for fid in similar_ids:
        phase_data = data_by_id[fid][phase]

        for point in phase_data:
            if round(point["minutes_since_start"]) == round(t_min):
                if "gspeed" in point and point["gspeed"] is not None:
                    speeds.append(point["gspeed"])
                break

    if not speeds:
        return None

    mean_speed_knots = sum(speeds) / len(speeds)

    # Convert knots → m/s
    return mean_speed_knots * 0.514444

In [ ]:
import matplotlib.pyplot as plt
import math

# ================================================
# CONFIGURATION EXPERIMENTALE
# ================================================
EXPERIMENTS = [
    ("39ba2570", "take_off"),
    ("39ba2570", "cruising"),
    ("39ba2570", "landing"),
]

K = 6          # number of similar flights to retrieve

def get_mean_similar_flights_position(similar_ids, data_by_id, phase, t_min):
    """
    Computes the mean latitude and longitude of the similar flights at the point
    closest to t_min in terms of minutes_since_start.
    """

    latitudes = []
    longitudes = []

    for fid in similar_ids:
        flight = data_by_id.get(fid)
        if not flight:
            continue

        phase_data = flight.get(phase)
        if not phase_data or len(phase_data) == 0:
            continue

        # Find the point in the similar flight closest to t_min
        closest_point = min(
            phase_data,
            key=lambda p: abs(p["minutes_since_start"] - t_min)
        )

        lat = closest_point.get("lat", None)
        lon = closest_point.get("lon", None)

        if lat is not None and lon is not None:
            latitudes.append(lat)
            longitudes.append(lon)

    # If no valid positions were found
    if len(latitudes) == 0:
        return (None, None)

    # Compute mean
    mean_lat = sum(latitudes) / len(latitudes)
    mean_lon = sum(longitudes) / len(longitudes)

    return [mean_lat, mean_lon, t_min]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


def predict_flight(flight_id, phase,
                   with_feedback_loop=True,
                   k=K,
                   batch_size=None,
                   use_physical_constraint=False,
                   epsilon=0.3):

    # --------------------------
    # 1️⃣ Extract target flight's data
    # --------------------------
    target_flight = data_by_id[flight_id]
    phase_data = target_flight[phase]
    aircraft_type = target_flight["flight_metadata"]["type"]

    # --------------------------
    # 2️⃣ Define spoofing zone
    # --------------------------
    N = len(phase_data)
    SPOOF_INDEX = N // 2

    observed_data = phase_data[:SPOOF_INDEX]
    spoofed_data  = phase_data[SPOOF_INDEX:]

    # --------------------------
    # 3️⃣ Generate partial summary
    # --------------------------
    partial_summary = generate_summary(phase, observed_data, aircraft_type, flight_id)

    predicted_positions = []
    ground_truth_positions = []
    mean_similar_flights_position = []

    for i, point in enumerate(spoofed_data):

        t_min = point["minutes_since_start"]
        barometric_altitude = point["altitude_ft"]

        # --------------------------
        # 4️⃣ Find similar flights
        # --------------------------
        altitude_prompt = f"""{t_min} minutes into the {phase} phase, the aircraft reached an altitude of {barometric_altitude} feet."""
        partial_summary = f"""{partial_summary} {altitude_prompt}"""

        neighbors = find_similar_flights(
            partial_summary,
            target_id=flight_id,
            phase=phase,
            k=k,
            batch_size=batch_size
        )

        similar_ids = [fid for fid, sim in neighbors]
        similar_summaries = [summary_by_phase[fid][phase] for fid in similar_ids]

        # --------------------------
        # 5️⃣ LLM Prediction
        # --------------------------
        if with_feedback_loop:
            prediction, num_token = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min,
                prediction_history=predicted_positions
            )
        else:
            prediction, num_token = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min
            )

        if prediction:

            pred_lat = float(prediction["lat"])
            pred_lon = float(prediction["lon"])

            # --------------------------
            # 🔹 OPTIONAL Physical Constraint
            # --------------------------
            if use_physical_constraint:

                if len(predicted_positions) > 0:
                    prev_lat, prev_lon, prev_time = predicted_positions[-1]
                else:
                    prev_point = observed_data[-1]
                    prev_lat = prev_point["lat"]
                    prev_lon = prev_point["lon"]
                    prev_time = prev_point["minutes_since_start"]

                delta_minutes = t_min - prev_time
                delta_seconds = delta_minutes * 60

                mean_speed = get_mean_speed_similar_flights(
                    similar_ids,
                    data_by_id,
                    phase,
                    prev_time
                )

                if mean_speed is not None and delta_seconds > 0:
                    expected_distance = mean_speed * delta_seconds

                    pred_lat, pred_lon, correction_status = enforce_ring_constraint_cartesian(
                        prev_lat,
                        prev_lon,
                        pred_lat,
                        pred_lon,
                        expected_distance,
                        epsilon
                    )

            predicted_positions.append([pred_lat, pred_lon, t_min])

        # --------------------------
        # Ground Truth
        # --------------------------
        ground_truth_positions.append([point["lat"], point["lon"], t_min])

        mean_similar_flights_position.append(
            get_mean_similar_flights_position(
                similar_ids,
                data_by_id,
                phase,
                t_min
            )
        )

    # --------------------------
    # Error computation
    # --------------------------
    mean_error = 0
    mean_similar_flight_error = 0

    for i in range(len(predicted_positions)):

        lat1, lon1, _ = predicted_positions[i]
        lat2, lon2, _ = ground_truth_positions[i]

        mean_error += haversine(lat1, lon1, lat2, lon2)

        lat1 = mean_similar_flights_position[i][0]
        lon1 = mean_similar_flights_position[i][1]

        mean_similar_flight_error += haversine(lat1, lon1, lat2, lon2)

    mean_error /= len(predicted_positions)
    mean_similar_flight_error /= len(mean_similar_flights_position)

    return (
        observed_data,
        ground_truth_positions,
        predicted_positions,
        round(mean_error / 1000, 1),
        mean_similar_flights_position,
        round(mean_similar_flight_error / 1000, 1),
        num_token
    )

# ================================================
# Principal experiment's function
# ================================================
def run_experiment_for_flight(flight_id, phase, with_feedback_loop: bool = False, json_path="results_RAG.json"):
    print("\n=================================================")
    print(f"🚀 EXPÉRIMENTATION SUR VOL {flight_id} | PHASE = {phase}")
    print("=================================================\n")

    observed_data, ground_truth_positions, predicted_positions, mean_error, mean_similar_flights_position, _, _ = predict_flight(flight_id, phase, with_feedback_loop=True, use_physical_constraint=False)

    # save_RAG_results(
    #     json_path=json_path,
    #     phase_name=phase,
    #     observed=observed_data,
    #     ground_truth=ground_truth_positions,
    #     predicted=predicted_positions,
    #     mean_error=mean_error,
    #     mean_similar_flight=mean_similar_flights_position
    # )

    m = plot_results(
        flight_id,
        phase,
        ground_truth_positions,
        predicted_positions,
        observed_data,
        mean_similar_flights_position
    )
    return m

def plot_results_folium(
        flight_id,
        phase,
        ground_truth_positions,
        predicted_positions,
        observed_data,
        mean_similar_flights_position
    ):
    # ---------------------------------------------------------
    # 1) Collect all lat/lon for map centering
    # ---------------------------------------------------------
    all_lats = [p["lat"] for p in observed_data] + \
               [p[0] for p in ground_truth_positions] + \
               [p[0] for p in predicted_positions] + \
               [p[0] for p in mean_similar_flights_position]

    all_lons = [p["lon"] for p in observed_data] + \
               [p[1] for p in ground_truth_positions] + \
               [p[1] for p in predicted_positions] + \
               [p[1] for p in mean_similar_flights_position]

    center_lat = sum(all_lats) / len(all_lats)
    center_lon = sum(all_lons) / len(all_lons)

    # ---------------------------------------------------------
    # 2) Create Folium map
    # ---------------------------------------------------------
    m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

    # ---------------------------------------------------------
    # 3) Build timestamps
    # ---------------------------------------------------------
    observed_times = [p["minutes_since_start"] for p in observed_data]

    base_time = observed_data[-1]["minutes_since_start"] if len(observed_data) > 0 else 0
    spoof_times = [base_time + (i + 1) for i in range(len(ground_truth_positions))]

    # ---------------------------------------------------------
    # 4) Plot observed trajectory (GREEN)
    # ---------------------------------------------------------
    obs_latlon = [(p["lat"], p["lon"]) for p in observed_data]
    folium.PolyLine(obs_latlon, color="green", weight=3, tooltip="Observed").add_to(m)

    for (lat, lon), t in zip(obs_latlon, observed_times):
        folium.Marker(
            location=(lat, lon),
            popup=f"Observed — {t:.1f} min",
            icon=folium.Icon(color="green")
        ).add_to(m)

    # ---------------------------------------------------------
    # 5) Ground truth trajectory (BLUE)
    # ---------------------------------------------------------
    gt_latlon = [(p[0], p[1]) for p in ground_truth_positions]
    folium.PolyLine(gt_latlon, color="blue", weight=3, tooltip="Ground Truth").add_to(m)

    for (lat, lon), t in zip(gt_latlon, spoof_times):
        folium.Marker(
            location=(lat, lon),
            popup=f"Ground Truth — {t:.1f} min",
            icon=folium.Icon(color="blue")
        ).add_to(m)

    # ---------------------------------------------------------
    # 6) Predicted trajectory (RED)
    # ---------------------------------------------------------
    pred_latlon = [(p[0], p[1]) for p in predicted_positions]
    folium.PolyLine(pred_latlon, color="red", weight=3, tooltip="Predicted").add_to(m)

    for (lat, lon), t in zip(pred_latlon, spoof_times):
        folium.Marker(
            location=(lat, lon),
            popup=f"Predicted — {t:.1f} min",
            icon=folium.Icon(color="red")
        ).add_to(m)

    # ---------------------------------------------------------
    # 7) Mean similar flights trajectory (ORANGE)
    # ---------------------------------------------------------
    mean_latlon = [(p[0], p[1]) for p in mean_similar_flights_position]
    folium.PolyLine(mean_latlon, color="orange", weight=3, tooltip="Mean Similar Flights").add_to(m)

    for (lat, lon), t in zip(mean_latlon, spoof_times):
        folium.Marker(
            location=(lat, lon),
            popup=f"Mean Similar — {t:.1f} min",
            icon=folium.Icon(color="orange")
        ).add_to(m)

    # ---------------------------------------------------------
    # 8) Add title (HTML displayed on map)
    # ---------------------------------------------------------
    title_html = f"""
        <h3 align="center" style="font-size:18px">
        Flight {flight_id} – Phase {phase}<br>
        Observed vs Ground Truth vs LLM Prediction vs Similar Flights
        </h3>
    """
    m.get_root().html.add_child(folium.Element(title_html))

    return m

def plot_results(
        flight_id,
        phase,
        ground_truth_positions,
        predicted_positions,
        observed_data,
        mean_similar_flights_position
    ):
    plt.figure(figsize=(12, 10))

    # --------------------------------
    # Build timestamps
    # --------------------------------
    observed_times = [p["minutes_since_start"] for p in observed_data]

    # --------------------------------
    # Observed trajectory (GREEN)
    # --------------------------------
    obs_lats = [p["lat"] for p in observed_data]
    obs_lons = [p["lon"] for p in observed_data]

    plt.plot(obs_lons, obs_lats, color="green", linewidth=2, label="Observed (before spoofing)")
    plt.scatter(obs_lons, obs_lats, color="green", s=40)

    for lon, lat, t in zip(obs_lons, obs_lats, observed_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="green")

    # --------------------------------
    # Ground truth trajectory (BLUE)
    # --------------------------------
    gt_lats = [p[0] for p in ground_truth_positions]
    gt_lons = [p[1] for p in ground_truth_positions]
    ground_truth_times = [p[2] for p in ground_truth_positions]

    plt.plot(gt_lons, gt_lats, color="blue", linewidth=2, label="Ground Truth (spoofed region)")
    plt.scatter(gt_lons, gt_lats, color="blue", s=40)

    for lon, lat, t in zip(gt_lons, gt_lats, ground_truth_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="blue")

    # --------------------------------
    # Predicted trajectory (RED)
    # --------------------------------
    pred_lats = [p[0] for p in predicted_positions]
    pred_lons = [p[1] for p in predicted_positions]
    spoof_times = [p[2] for p in predicted_positions]

    plt.plot(pred_lons, pred_lats, color="red", linewidth=2, label="Predicted trajectory (LLM)")
    plt.scatter(pred_lons, pred_lats, color="red", s=40)

    for lon, lat, t in zip(pred_lons, pred_lats, spoof_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="red")

    # --------------------------------
    # Mean similar flights trajectory (ORANGE)
    # --------------------------------
    mean_lats = [p[0] for p in mean_similar_flights_position]
    mean_lons = [p[1] for p in mean_similar_flights_position]
    spoof_times = [p[2] for p in predicted_positions]

    plt.plot(mean_lons, mean_lats, color="orange", linewidth=2, linestyle="--",
             label="Mean similar flights trajectory")
    plt.scatter(mean_lons, mean_lats, color="orange", s=40)

    for lon, lat, t in zip(mean_lons, mean_lats, spoof_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="orange")

    # --------------------------------
    # Final layout
    # --------------------------------
    plt.title(
        f"Flight {flight_id} – Phase {phase}\n"
        f"Observed vs Ground Truth vs LLM Prediction",
        fontsize=14
    )
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
for flight_id, phase in EXPERIMENTS:
    m = run_experiment_for_flight(flight_id, phase, with_feedback_loop=True)

# m.save("map_output.html")

NOW LETS CALCULATE THE AVERAGE ERROR OVER THE 100 VALIDATION TEST

In [ ]:
import json

# Path to your JSON file
input_file = "data/CDG-FCO/lstm_validation_flight_ids.json"

with open(input_file, "r", encoding="utf-8") as f:
    validation_flight_ids = json.load(f)

print(f"Loaded {len(validation_flight_ids)} flight IDs from {input_file}")

results = {"take_off": 0, "cruising": 0, "landing": 0}

In [ ]:
import random

def calculate_mean_haversine_error(
    flight_ids_by_phase,
    k=5,
    batch_size=None
):
    errors_takeoff = []
    errors_cruising = []
    errors_landing = []

    similar_flight_errors_takeoff = []
    similar_flight_errors_cruising = []
    similar_flight_errors_landing = []

    mean_num_token_takeoff = 0
    mean_num_token_cruising = 0
    mean_num_token_landing = 0

    counter_takeoff = 0
    counter_cruising = 0
    counter_landing = 0

    # ======================================================
    # TAKE OFF
    # ======================================================
    for flight_id in flight_ids_by_phase["take_off"]:
        try:
            _, _, _, error, _, similar_error, num_tokens = predict_flight(
                flight_id, "take_off",
                with_feedback_loop=True,
                k=k,
                batch_size=batch_size
            )
            errors_takeoff.append((error, flight_id))
            similar_flight_errors_takeoff.append(similar_error)
            mean_num_token_takeoff += num_tokens
            counter_takeoff += 1
            print(f"counter take off: {counter_takeoff}")
        except:
            continue

    # ======================================================
    # CRUISING
    # ======================================================
    for flight_id in flight_ids_by_phase["cruising"]:
        try:
            _, _, _, error, _, similar_error, num_tokens = predict_flight(
                flight_id, "cruising",
                with_feedback_loop=True,
                k=k,
                batch_size=batch_size
            )
            errors_cruising.append((error, flight_id))
            similar_flight_errors_cruising.append(similar_error)
            mean_num_token_cruising += num_tokens
            counter_cruising += 1
            print(f"counter cruising: {counter_cruising}")
        except:
            continue

    # ======================================================
    # LANDING
    # ======================================================
    for flight_id in flight_ids_by_phase["landing"]:
        try:
            _, _, _, error, _, similar_error, num_tokens = predict_flight(
                flight_id, "landing",
                with_feedback_loop=True,
                k=k,
                batch_size=batch_size
            )
            errors_landing.append((error, flight_id))
            similar_flight_errors_landing.append(similar_error)
            mean_num_token_landing += num_tokens
            counter_landing += 1
            print(f"counter landing: {counter_landing}")
        except:
            continue

    # ======================================================
    # Means
    # ======================================================
    mean_num_token_takeoff = (
        mean_num_token_takeoff / counter_takeoff if counter_takeoff > 0 else None
    )
    mean_num_token_cruising = (
        mean_num_token_cruising / counter_cruising if counter_cruising > 0 else None
    )
    mean_num_token_landing = (
        mean_num_token_landing / counter_landing if counter_landing > 0 else None
    )

    return (
        errors_takeoff,
        similar_flight_errors_takeoff,
        errors_cruising,
        similar_flight_errors_cruising,
        errors_landing,
        similar_flight_errors_landing,
        mean_num_token_takeoff,
        mean_num_token_cruising,
        mean_num_token_landing
    )


In [ ]:
with open("../data/CDG-FCO/RESULTS/test_sample.json", "r", encoding="utf-8") as f:
    flight_ids_by_phase = json.load(f)

errors_takeoff, similar_flight_error_takeoff, errors_cruising, similar_flight_error_cruising, errors_landing, similar_flight_error_landing, _, _, _ = calculate_mean_haversine_error(flight_ids_by_phase, k=5)

results = {
    "RAG": 
           {"take_off": errors_takeoff, 
            "cruising": errors_cruising, 
            "landing": errors_landing}, 
    "MEAN_SIMILAR_FLIGHT": 
        {"take_off": similar_flight_error_takeoff, 
         "cruising": similar_flight_error_cruising, 
         "landing": similar_flight_error_landing}
}

In [ ]:
import json

output_file = "../data/MULTI_ROUTE/RESULTS/MEAN_HAVERSINE_RAG.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"✅ Saved to {output_file}")

In [ ]:
import json
import numpy as np

# ==========================================
# Path to your results file
# ==========================================
JSON_PATH = "../data/MULTI_ROUTE/RESULTS/MEAN_HAVERSINE_RAG.json"   # <-- change if needed

# ==========================================
# Load JSON
# ==========================================
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

rag_results = data["RAG"]

PHASES = ["take_off", "cruising", "landing"]

print("\n📊 RAG Statistical Results\n")

for phase in PHASES:
    # Extract only the error values (first element of each pair)
    errors = [item[0] for item in rag_results[phase]]

    errors = np.array(errors)

    mean_val   = np.mean(errors)
    median_val = np.median(errors)
    std_val    = np.std(errors)

    print(f"🔹 {phase.upper()}")
    print(f"   Mean   : {mean_val:.2f} km")
    print(f"   Median : {median_val:.2f} km")
    print(f"   Std    : {std_val:.2f} km\n")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Load data
# =========================
FILEPATH = "../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_RAG.json"

with open(FILEPATH, "r", encoding="utf-8") as f:
    data = json.load(f)

phases = ["take_off", "cruising", "landing"]
rag = data["RAG"]

# Prepare arrays in the same order as phases
values = [np.array(rag[p], dtype=float) for p in phases]

# =========================
# Stats (mean/median/std)
# =========================
stats = {
    p: {
        "n": len(values[i]),
        "mean": float(np.mean(values[i])),
        "median": float(np.median(values[i])),
        "std": float(np.std(values[i], ddof=1)) if len(values[i]) > 1 else float("nan"),
    }
    for i, p in enumerate(phases)
}

print("Summary stats (RAG):")
for p in phases:
    s = stats[p]
    print(f"- {p}: n={s['n']}, mean={s['mean']:.3f}, median={s['median']:.3f}, std={s['std']:.3f}")

# =========================
# Box plot
# =========================
fig, ax = plt.subplots(figsize=(9, 5))

bp = ax.boxplot(
    values,
    labels=phases,
    showmeans=True,     # shows mean marker
    meanline=False,     # set True if you prefer a mean line
    whis=1.5            # standard whiskers: 1.5 * IQR
)

ax.set_title("TrajRAG Mean Haversine Error Distribution per Flight Phase")
ax.set_ylabel("Haversine error (km)")
ax.grid(axis="y", alpha=0.3)

# Add mean/median text above each box
ymax = max(v.max() for v in values if len(v) > 0)
ymin = min(v.min() for v in values if len(v) > 0)
yrange = ymax - ymin if ymax > ymin else 1.0

for i, p in enumerate(phases, start=1):
    s = stats[p]
    txt = f"mean={s['mean']:.2f}\nmed={s['median']:.2f}"
    ax.text(i, ymax + 0.03 * yrange, txt, ha="center", va="bottom", fontsize=9)

ax.set_ylim(top=ymax + 0.18 * yrange)

plt.tight_layout()
plt.show()


LETS VARIATE THE VARIABLE K WHICH IS THE NUMBER OF SIMILAR FLIGHTS DURING THE RETRIEVAL PHASE OF THE RAG PIPELINE

In [ ]:
# Path to your JSON file

k_values = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
with open("../data/CDG-FCO/RESULTS/test_sample.json", "r", encoding="utf-8") as f:
    flight_ids_by_phase = json.load(f)

results = {}

for k in k_values:
    print(f"The number of similar flight during retrieval is {k}")
    errors_takeoff, _, errors_cruising, _, errors_landing, _, mean_num_token_takeoff, mean_num_token_cruising, mean_num_token_landing = calculate_mean_haversine_error(flight_ids_by_phase, k=k)
    results[k] = {
        "take_off": {"error": np.mean(errors_takeoff), "num_token": mean_num_token_takeoff}, 
        "cruising": {"error": np.mean(errors_cruising), "num_token": mean_num_token_cruising},
        "landing": {"error": np.mean(errors_landing), "num_token": mean_num_token_landing}
        }


In [ ]:
# ======================================================
# Save output
# ======================================================
with open("../data/CDG-FCO/RESULTS/K_COMPARISON.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

print(f"Saved random test sample to OUTPUT_FILE")

In [ ]:
import json
import matplotlib.pyplot as plt

# =========================
# Load JSON
# =========================
JSON_PATH = "data/error_vs_kneighbors.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

data = {int(k): v for k, v in raw.items()}
Ks = sorted(data.keys())
PHASES = ["take_off", "cruising", "landing"]

# =========================
# Plot: Tokens vs K (per phase)
# =========================
plt.figure(figsize=(10, 6))

for phase in PHASES:
    tokens = [data[k][phase]["num_token"] for k in Ks]
    plt.plot(Ks, tokens, marker="o", label=phase)

plt.xlabel("K (number of retrieved neighbors)")
plt.ylabel("Tokens sent to OpenAI (generation prompt)")
plt.title("Tokens vs K (per phase)")
plt.xticks(Ks)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


LETS COMPUTE THE MSE USING MORE AND MORE DATA AND SEE HOW FAST IT CAN CONVERGE

In [ ]:
import os
import json
import faiss
import numpy as np
from pathlib import Path
from openai import OpenAI

# --------------------------
# Config
# --------------------------
MODEL = "text-embedding-3-large"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))

PHASES = ["take_off", "cruising", "landing"]
BATCH_DIR = Path("data/EXPERIMENTS/MSE_CONVERGENCE_SPEED/batches")
OUT_DIR = Path("data/CDG-FCO/sub_indexes")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------
# Load global summaries and data
# --------------------------
print("📥 Loading summaries and data...")
with open("data/CDG-FCO/flight_summary_separated_into_phases.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

with open("data/CDG-FCO/flight_data_with_minutes_since_start.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

# Map fr24_id -> raw data per phase
data_by_id = {f["flight_metadata"]["fr24_id"]: f for f in flights_data}

# Map fr24_id -> summaries per phase
summary_by_phase = {}
for s in summaries:
    fid = s["fr24_id"]
    summary_by_phase[fid] = {}
    for phase in PHASES:
        text = s.get(phase)
        summary_by_phase[fid][phase] = text if isinstance(text, str) and text.strip() else None

common_ids = list(set(summary_by_phase.keys()) & set(data_by_id.keys()))
print(f"✅ Common flights: {len(common_ids)}")

# --------------------------
# Build FAISS sub-index per phase and batch
# --------------------------
BATCH_SIZES = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 831]

In [ ]:
# --------------------------
# Embeddings helper
# --------------------------
def get_embeddings(texts, model=MODEL, batch_size=64):
    vectors = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        batch_vecs = [np.array(e.embedding, dtype="float32") for e in resp.data]
        vectors.extend(batch_vecs)
    X = np.vstack(vectors)
    X = X / np.linalg.norm(X, axis=1, keepdims=True)  # L2 normalization
    return X

for batch_size in BATCH_SIZES:
    print(batch_size)
    batch_file = BATCH_DIR / f"batch_ids_{batch_size}.json"
    if not batch_file.exists():
        print(f"⚠️ Missing batch file {batch_file}, skipping...")
        continue

    with open(batch_file, "r", encoding="utf-8") as f:
        batch_ids = set(json.load(f))

    print(f"\n==============================")
    print(f"🚀 Building sub-index for batch size = {batch_size}")
    print(f"==============================")

    for phase in PHASES:
        print(f"\n🧠 Phase: {phase}")

        # Filter flights belonging to this batch that have a summary
        valid_flights = [
            fid for fid in batch_ids
            if fid in summary_by_phase and summary_by_phase[fid].get(phase)
        ]

        if not valid_flights:
            print(f"⚠️ No valid flights for {phase} at batch size {batch_size}, skipping.")
            continue

        texts = [summary_by_phase[fid][phase] for fid in valid_flights]
        print(f"📄 Computing embeddings for {len(valid_flights)} valid flights...")

        # Compute embeddings
        E = get_embeddings(texts)
        d = E.shape[1]

        # Build FAISS index
        index = faiss.IndexFlatIP(d)
        index.add(E)

        # Save outputs
        emb_out = OUT_DIR / f"embeddings_{phase}_{batch_size}.npy"
        idx_out = OUT_DIR / f"faiss_index_{phase}_{batch_size}.index"
        ids_out = OUT_DIR / f"ids_dictionary_{phase}_{batch_size}.json"

        np.save(emb_out, E)
        faiss.write_index(index, str(idx_out))
        with open(ids_out, "w", encoding="utf-8") as f:
            json.dump(valid_flights, f, ensure_ascii=False, indent=2)

        print(f"✅ Saved {phase} batch={batch_size}: {len(valid_flights)} embeddings → {idx_out.name}")

print("\n🎯 All sub-indexes built successfully!")


In [ ]:
# Path to your JSON file

input_file = "data/CDG-FCO/lstm_validation_flight_ids.json"

with open(input_file, "r", encoding="utf-8") as f:
    validation_flight_ids = json.load(f)

print(f"Loaded {len(validation_flight_ids)} flight IDs from {input_file}")
results = {}

for bs in BATCH_SIZES:
    print(f"The batch size is {bs}")
    input_file = f"data/EXPERIMENTS/MSE_CONVERGENCE_SPEED/batches/batch_ids_{bs}.json"
    with open(input_file, "r", encoding="utf-8") as f:
        batch_ids = json.load(f)
    summary_by_phase = {
        s["fr24_id"]: {p: s.get(p, None) for p in ["take_off", "cruising", "landing"]}
        for s in summaries if s["fr24_id"] in batch_ids
    }
    
    mean_error_takeoff, _, mean_error_cruising, _, mean_error_landing, _, _, _, _ = calculate_mean_haversine_error(validation_flight_ids, k=6, batch_size=bs)
    results[bs] = {
        "take_off": {"error": mean_error_takeoff}, 
        "cruising": {"error": mean_error_cruising},
        "landing": {"error": mean_error_landing}
        }

In [ ]:
# If your dict has non-string keys (like ints), JSON will convert them to strings.
with open("experiments/sample_efficiency/results_RAG.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)